# Dubai VARA-Licensed Crypto Exchange Comparison & Scoring Tool


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Dubai VARA-Licensed Crypto Exchange Comparison & Scoring Tool

Interactive tool to evaluate and rank VARA-licensed cryptocurrency exchanges available to UAE traders. Based on analysis of 7 major platforms reviewed for compliance, security, and local trading support.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Exchange dataset: 7 VARA-licensed platforms with key attributes
exchanges_data = {
    'Exchange': ['Kraken', 'Coinbase', 'Bybit', 'OKX', 'Crypto.com', 'Gemini', 'Kucoin'],
    'VARA_Licensed': [1, 1, 1, 1, 1, 1, 1],  # All VARA-regulated
    'AED_Trading_Pairs': [15, 12, 8, 20, 18, 10, 14],  # Number of AED pairs
    'Security_Audit': [1, 1, 1, 1, 0, 1, 0],  # Third-party security audit (1=yes, 0=no)
    'Founded_Year': [2011, 2012, 2020, 2017, 2016, 2015, 2017],
    'Maker_Fee_Percent': [0.16, 0.4, 0.1, 0.02, 0.4, 0.25, 0.1],
    'UAE_Customer_Support': [1, 1, 1, 1, 1, 1, 0.5],  # 1=full 24/7 Arabic, 0.5=partial
    'Fiat_Onramp_AED': [1, 1, 0, 1, 1, 0, 0]
}

df = pd.DataFrame(exchanges_data)
print("Exchange Attributes Dataset:")
print(df)
print("\nDataset loaded. 7 VARA-licensed exchanges ready for analysis.")

## Weighted Scoring System

Create a composite score weighing factors important to UAE traders: VARA compliance (mandatory), AED pair availability, security audits, customer support, and fiat on-ramp accessibility.

In [ ]:
# Define scoring weights (normalized to sum = 100)
weights = {
    'VARA_weight': 0.25,        # Regulatory compliance critical
    'AED_pairs_weight': 0.20,   # Local currency trading convenience
    'Security_weight': 0.20,    # Third-party audit confirmation
    'Support_weight': 0.15,     # UAE-specific customer service
    'Onramp_weight': 0.10,      # Direct AED deposit/withdrawal
    'Fee_weight': 0.10          # Competitive trading costs (inverse)
}

# Normalize metrics to 0-100 scale for scoring
df['VARA_Score'] = df['VARA_Licensed'] * 100  # All licensed = 100
df['AED_Score'] = (df['AED_Trading_Pairs'] / df['AED_Trading_Pairs'].max()) * 100
df['Security_Score'] = df['Security_Audit'] * 100
df['Support_Score'] = df['UAE_Customer_Support'] * 100
df['Onramp_Score'] = df['Fiat_Onramp_AED'] * 100
df['Fee_Score'] = 100 - (df['Maker_Fee_Percent'] / df['Maker_Fee_Percent'].max()) * 100

# Calculate composite score
df['Composite_Score'] = (
    df['VARA_Score'] * weights['VARA_weight'] +
    df['AED_Score'] * weights['AED_pairs_weight'] +
    df['Security_Score'] * weights['Security_weight'] +
    df['Support_Score'] * weights['Support_weight'] +
    df['Onramp_Score'] * weights['Onramp_weight'] +
    df['Fee_Score'] * weights['Fee_weight']
).round(2)

# Rank exchanges
df['Rank'] = df['Composite_Score'].rank(ascending=False, method='min').astype(int)

ranking = df[['Rank', 'Exchange', 'Composite_Score', 'AED_Trading_Pairs', 'Security_Audit', 'Maker_Fee_Percent']].sort_values('Rank')
print("\nExchange Rankings (Weighted for UAE Traders):")
print(ranking.to_string(index=False))

## Visualization: Exchange Comparison Heatmap

Visual comparison of all exchanges across key evaluation criteria.

In [ ]:
# Create heatmap of normalized scores
score_cols = ['VARA_Score', 'AED_Score', 'Security_Score', 'Support_Score', 'Onramp_Score', 'Fee_Score']
heatmap_data = df[['Exchange'] + score_cols].set_index('Exchange')
heatmap_data.columns = ['VARA Compliance', 'AED Pairs', 'Security Audit', 'Support Quality', 'Fiat On-ramp', 'Fee Structure']

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn', cbar_kws={'label': 'Score (0-100)'}, ax=ax, vmin=0, vmax=100)
plt.title('Dubai VARA-Licensed Exchanges: Multi-Factor Comparison Heatmap', fontsize=14, fontweight='bold')
plt.ylabel('Exchange')
plt.xlabel('Evaluation Criteria')
plt.tight_layout()
plt.show()

print("\nHeatmap generated: Green = strong performance, Red = gaps")

## Interactive Decision Matrix

Filter exchanges by trader priorities: budget sensitivity, security requirement, or local trading convenience.

In [ ]:
# Decision filter functions
def filter_by_budget(df, max_maker_fee=0.20):
    """Filter exchanges for cost-conscious traders."""
    return df[df['Maker_Fee_Percent'] <= max_maker_fee][['Exchange', 'Maker_Fee_Percent', 'Composite_Score']].sort_values('Composite_Score', ascending=False)

def filter_by_security(df, require_audit=True):
    """Filter exchanges that passed third-party security audits."""
    filtered = df[df['Security_Audit'] == (1 if require_audit else 0)]
    return filtered[['Exchange', 'Security_Audit', 'Composite_Score']].sort_values('Composite_Score', ascending=False)

def filter_by_local_trading(df, min_aed_pairs=10):
    """Filter for traders prioritizing AED pair availability."""
    return df[df['AED_Trading_Pairs'] >= min_aed_pairs][['Exchange', 'AED_Trading_Pairs', 'Fiat_Onramp_AED', 'Composite_Score']].sort_values('Composite_Score', ascending=False)

print("\n=== SCENARIO 1: Cost-Conscious Trader (Maker Fee <= 0.20%) ===")
print(filter_by_budget(df))

print("\n=== SCENARIO 2: Security-First Requirement (Third-Party Audit) ===")
print(filter_by_security(df, require_audit=True))

print("\n=== SCENARIO 3: Local AED Trading Priority (>= 10 AED pairs) ===")
print(filter_by_local_trading(df, min_aed_pairs=10))

## Summary & Recommendation

Run the cells above to:
- Rank all 7 VARA-licensed exchanges by weighted composite score
- Visualize strengths/weaknesses across compliance, fees, security, and local support
- Filter exchanges matching your trader profile (budget, security, local convenience)

All platforms analyzed are VARA-regulated and compliant with Dubai Financial Services Authority requirements.

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/cryptocurrency-exchange-dubai-reviews)
